# RetailVelocity — End-to-End Walkthrough

This notebook runs the full analytic pipeline on the synthetic dataset:

1. **Ingestion** with Polars `scan_parquet` (lazy).
2. **Descriptive** — revenue over time, by category, by country, weekday × hour heatmap.
3. **Diagnostic (RFM)** — customer segmentation into Platinum/Gold/Silver/Bronze.
4. **Cohort** — acquisition retention matrix.
5. **Predictive** — SKU-level demand forecast with Holt-Winters.
6. **Prescriptive** — reorder points and the at-risk dollar total.
7. **Benchmarks** — Polars eager vs lazy throughput.

## 0. Ensure the dataset exists

Run `uv run retailvelocity generate` from the project root once. Or call the generator inline:

In [ ]:
from retailvelocity.config import TRANSACTIONS_PARQUET

if not TRANSACTIONS_PARQUET.exists():
    from retailvelocity.data_gen import GenConfig, generate
    generate(GenConfig())  # defaults: 1M rows, 50k customers, 2k products, 3 years

TRANSACTIONS_PARQUET

## 1. Ingestion — lazy

In [ ]:
import polars as pl
from retailvelocity.ingestion import dataset_summary, load_enriched

lf = load_enriched()  # LazyFrame with tx + customer + product joined
print(lf.collect_schema())
dataset_summary(lf)

## 2. Descriptive

In [ ]:
from retailvelocity.descriptive import (
    revenue_by_category,
    revenue_by_country,
    rolling_revenue,
    weekday_hour_heatmap,
)

revenue_by_category(lf).collect()

In [ ]:
revenue_by_country(lf).collect()

In [ ]:
rolling_revenue(lf, window_days=7, grain='day').collect().tail(10)

In [ ]:
weekday_hour_heatmap(lf).collect().head(10)

## 3. RFM segmentation

In [ ]:
from retailvelocity.rfm import at_risk_customers, compute_rfm, tier_summary

rfm = compute_rfm(lf).collect()
rfm.head()

In [ ]:
tier_summary(rfm.lazy()).collect()

In [ ]:
at_risk = at_risk_customers(rfm.lazy()).collect()
print(f'{at_risk.height} win-back candidates, ${at_risk["monetary"].sum():,.0f} at risk')
at_risk.head()

## 4. Cohort retention

In [ ]:
from retailvelocity.cohort import cohort_heatmap_pivot, monthly_cohort_matrix

matrix = monthly_cohort_matrix(lf)
cohort_heatmap_pivot(matrix, max_periods=12)

## 5. Forecasting — top SKUs

In [ ]:
from retailvelocity.forecasting import forecast_product, top_skus_by_revenue

ids = top_skus_by_revenue(lf, n=5)
result = forecast_product(lf, ids[0], horizon_days=30, test_days=30)
print(f'{result.sku}  MAPE={result.mape:.1f}%  RMSE={result.rmse:.2f}')
result.forecast.head()

## 6. Prescriptive — reorder + dead stock

In [ ]:
from retailvelocity.forecasting import forecast_many
from retailvelocity.ingestion import load_products
from retailvelocity.prescriptive import dead_stock, reorder_report

products = load_products().collect()
forecasts = forecast_many(lf, top_skus_by_revenue(lf, n=20), horizon_days=14)
reorder = reorder_report(forecasts, products, lead_time_days=14)
reorder

In [ ]:
dead = dead_stock(lf, products, lookback_days=180)
print(f'{dead.height} dead-stock SKUs, ${dead["parked_value"].sum():,.0f} parked')
dead.head()

## 7. Performance — Polars eager vs lazy

In [ ]:
from retailvelocity.benchmarks import run_all

run_all()